In [1]:
import sys, os
import json
import re
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from src.utils.safe_loader import safe_load_npy
from src.training.scaler import FeatureScaler
from src.training.feature_selection import (
    get_feature_names, 
    run_feature_selection, 
    save_selected_features,
    get_logistic_regression
)

FIGURES_DIR = 'figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

PHASE_DIR = os.path.join(
    PROJECT_ROOT,
    '.planning', 'phases', '03-dataset-standardization-mrmr-selection'
)

print('Setup complete.')
print(f'Project root: {PROJECT_ROOT}')
print(f'CWD: {os.getcwd()}')


Setup complete.
Project root: /run/media/mananbyte/newvol/Pannuke-project
CWD: /run/media/mananbyte/newvol/Pannuke-project/notebooks


## Extract Winning Scaler from DECISION.md

In [2]:
decision_md_path = os.path.join(PHASE_DIR, 'DECISION.md')
with open(decision_md_path, 'r') as fh:
    content = fh.read()

match = re.search(r'\*\*Final choice:\*\* `([^`]+)`', content)
if match:
    winning_scaler = match.group(1)
else:
    winning_scaler = 'RobustScaler'

print(f'Extracted winning scaler: {winning_scaler}')


Extracted winning scaler: RobustScaler


## Section 1 — Setup & Scaled Data Loading

In [3]:
X_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'fold1_binary_X.npy')
Y_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'fold1_binary_y.npy')

print('Loading raw feature matrix (mmap, read-only)...')
X_mm = safe_load_npy(X_PATH, mode='r')
y_mm = safe_load_npy(Y_PATH, mode='r')

print(f'X shape: {X_mm.shape}, y shape: {y_mm.shape}')

SAMPLE_N = 100_000
RANDOM_STATE = 42

print(f'Drawing stratified subsample of {SAMPLE_N:,} rows...')
_, sub_idx = train_test_split(
    np.arange(len(y_mm)),
    test_size=SAMPLE_N,
    stratify=np.array(y_mm),
    random_state=RANDOM_STATE,
)
sub_idx = np.sort(sub_idx)
X_sub = np.array(X_mm[sub_idx], dtype=np.float32)
y_sub = np.array(y_mm[sub_idx], dtype=np.int32)
X_sub = np.nan_to_num(X_sub, nan=0.0, posinf=0.0, neginf=0.0)

print('Fitting extracted scaler and transforming subsample...')
fs = FeatureScaler(scaler_type=winning_scaler)
fs._scaler.fit(X_sub)
X_scaled = fs._scaler.transform(X_sub).astype(np.float32)
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# Split 80k train / 20k validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_scaled, y_sub, test_size=0.2, stratify=y_sub, random_state=RANDOM_STATE
)

print(f'Train shape: {X_tr.shape}, Val shape: {X_val.shape}')
classes, counts = np.unique(y_tr, return_counts=True)
print('Train class distribution:')
for c, n in zip(classes, counts):
    print(f'  class {c}: {n:,}  ({n/len(y_tr)*100:.1f}%)')


Loading raw feature matrix (mmap, read-only)...
X shape: (2073571, 93), y shape: (2073571,)
Drawing stratified subsample of 100,000 rows...


Fitting extracted scaler and transforming subsample...


Train shape: (80000, 93), Val shape: (20000, 93)
Train class distribution:
  class 0: 40,988  (51.2%)
  class 1: 39,012  (48.8%)


## Section 2 — N-Sweep: F1 vs Feature Count

In [4]:
N_VALUES = [10, 15, 20, 25, 30]
METHODS = ['mrmr', 'anova', 'rfe']

sweep_results = []
feature_selections = {}  # Store selected indices for correlation/Jaccard

for method in METHODS:
    feature_selections[method] = {}
    for n in N_VALUES:
        print(f'Running selection: method={method}, n={n}...')
        indices, names = run_feature_selection(X_tr, y_tr, n_features=n, method=method)
        feature_selections[method][n] = (indices, names)
        
        # Train proxy model
        lr = get_logistic_regression(max_iter=500, random_state=RANDOM_STATE)
        lr.fit(X_tr[:, indices], y_tr)
        y_pred = lr.predict(X_val[:, indices])
        f1 = float(f1_score(y_val, y_pred, average='macro'))
        
        sweep_results.append({
            'method': method,
            'n': n,
            'macro_f1': f1
        })
        print(f'  -> F1: {f1:.4f}')

# Fit full 93 features baseline
print('Running full 93 features baseline...')
lr_full = get_logistic_regression(max_iter=500, random_state=RANDOM_STATE)
lr_full.fit(X_tr, y_tr)
y_pred_full = lr_full.predict(X_val)
baseline_f1 = float(f1_score(y_val, y_pred_full, average='macro'))
print(f'Baseline F1 (93 features): {baseline_f1:.4f}')

df_sweep = pd.DataFrame(sweep_results)
df_pivot = df_sweep.pivot(index='n', columns='method', values='macro_f1')
print('\nSweep Results Table:')
print(df_pivot.to_string())


Running selection: method=mrmr, n=10...
Computing ANOVA F-statistic for relevance...
Computing correlation matrix for redundancy...
Running mRMR selection loop...


  -> F1: 0.8535
Running selection: method=mrmr, n=15...
Computing ANOVA F-statistic for relevance...
Computing correlation matrix for redundancy...
Running mRMR selection loop...


  -> F1: 0.8550
Running selection: method=mrmr, n=20...
Computing ANOVA F-statistic for relevance...
Computing correlation matrix for redundancy...
Running mRMR selection loop...


  -> F1: 0.8598
Running selection: method=mrmr, n=25...
Computing ANOVA F-statistic for relevance...
Computing correlation matrix for redundancy...
Running mRMR selection loop...


  -> F1: 0.8605
Running selection: method=mrmr, n=30...
Computing ANOVA F-statistic for relevance...
Computing correlation matrix for redundancy...
Running mRMR selection loop...


...

  -> F1: 0.8681
Running selection: method=rfe, n=30...


[2026-05-26 03:11:41.878] [CUML] [warning] L-BFGS: max iterations reached
[2026-05-26 03:11:41.878] [CUML] [warning] Maximum iterations reached before solver is converged. To increase model accuracy you can increase the number of iterations (max_iter) or improve the scaling of the input data.


[2026-05-26 03:11:44.073] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step


  -> F1: 0.8683
Running full 93 features baseline...


Baseline F1 (93 features): 0.8699

Sweep Results Table:
method     anova      mrmr       rfe
n                                   
10      0.853530  0.853479  0.861278
15      0.856173  0.855049  0.863513
20      0.855525  0.859813  0.867893
25      0.859730  0.860524  0.868138
30      0.860734  0.861577  0.868343


## Section 4 — Elbow Rule Application

In [5]:
# Elbow point logic: smallest n where delta_f1 < 0.002
ELBOW_THRESHOLD = 0.002
deltas = []

for method in METHODS:
    method_rows = df_sweep[df_sweep['method'] == method].sort_values('n')
    f1_list = method_rows['macro_f1'].tolist()
    n_list = method_rows['n'].tolist()
    
    # n=10 has no delta
    deltas.append({'method': method, 'n': 10, 'macro_f1': f1_list[0], 'delta_f1': 0.0})
    
    for idx in range(1, len(f1_list)):
        delta = f1_list[idx] - f1_list[idx-1]
        deltas.append({
            'method': method,
            'n': n_list[idx],
            'macro_f1': f1_list[idx],
            'delta_f1': delta
        })

df_deltas = pd.DataFrame(deltas)
print('Deltas Table:')
print(df_deltas.to_string(index=False))

# Find elbow point for each method
elbow_points = {}
for method in METHODS:
    m_deltas = df_deltas[df_deltas['method'] == method].sort_values('n')
    chosen_n = 20  # Default if no elbow is found
    # Scan from n=15 upwards
    for _, row in m_deltas.iterrows():
        if row['n'] == 10:
            continue
        if row['delta_f1'] < ELBOW_THRESHOLD:
            # This is the first n where improvement is below threshold
            # So we choose the *previous* n, or this n? The rule says:
            # "smallest n where the marginal F1 gain from adding the next 5 features drops below 0.002"
            # This means adding features beyond this n yields marginal returns < 0.002.
            # So we stop at this row's previous n, or this row's n itself as the point where the elbow occurred.
            # Let's select the row's n itself as the elbow feature count.
            chosen_n = int(row['n'])
            break
    elbow_points[method] = chosen_n

print('\nElbow points identified:')
for method, en in elbow_points.items():
    print(f'  {method}: n={en}')

# We will choose the overall winner method based on F1 at their respective elbow points
best_score = -1
best_method = None
best_n = None

for method, en in elbow_points.items():
    score = df_sweep[(df_sweep['method'] == method) & (df_sweep['n'] == en)]['macro_f1'].values[0]
    if score > best_score:
        best_score = score
        best_method = method
        best_n = en

print(f'\nELBOW RESULT: winner_method={best_method}, chosen_n={best_n}, validation_macro_f1={best_score:.4f}')


Deltas Table:
method  n  macro_f1  delta_f1
  mrmr 10  0.853479  0.000000
  mrmr 15  0.855049  0.001570
  mrmr 20  0.859813  0.004764
  mrmr 25  0.860524  0.000711
  mrmr 30  0.861577  0.001052
 anova 10  0.853530  0.000000
 anova 15  0.856173  0.002643
 anova 20  0.855525 -0.000648
 anova 25  0.859730  0.004205
 anova 30  0.860734  0.001004
   rfe 10  0.861278  0.000000
   rfe 15  0.863513  0.002235
   rfe 20  0.867893  0.004380
   rfe 25  0.868138  0.000245
   rfe 30  0.868343  0.000205

Elbow points identified:
  mrmr: n=15
  anova: n=20
  rfe: n=25

ELBOW RESULT: winner_method=rfe, chosen_n=25, validation_macro_f1=0.8681


## Section 3 — Elbow Plot (F1 vs n)

In [6]:
plt.figure(figsize=(10, 6))
for method in METHODS:
    m_rows = df_sweep[df_sweep['method'] == method].sort_values('n')
    plt.plot(m_rows['n'], m_rows['macro_f1'], marker='o', label=method.upper(), linewidth=2)
    # Draw dashed line at this method's elbow point
    plt.axvline(x=elbow_points[method], linestyle='--', alpha=0.5, label=f'{method.upper()} elbow (n={elbow_points[method]})')

plt.axhline(y=baseline_f1, color='grey', linestyle=':', label=f'Full Baseline (n=93, F1={baseline_f1:.4f})', alpha=0.8)

plt.title('Feature Count vs Proxy Macro-F1 — Elbow Analysis', fontsize=12)
plt.xlabel('Number of Features (n)', fontsize=10)
plt.ylabel('Proxy Model Validation Macro-F1', fontsize=10)
plt.xticks(N_VALUES)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right', fontsize=8)

plot_path = os.path.join(FIGURES_DIR, '07_f1_elbow.png')
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.close()
print(f'Elbow plot saved -> {plot_path}')


Elbow plot saved -> figures/07_f1_elbow.png


## Section 5 — Jaccard Overlap Heatmap

In [7]:
jaccard_matrix = np.zeros((len(METHODS), len(METHODS)))

for i, m1 in enumerate(METHODS):
    idx1 = set(feature_selections[m1][best_n][0])
    for j, m2 in enumerate(METHODS):
        idx2 = set(feature_selections[m2][best_n][0])
        intersection = len(idx1.intersection(idx2))
        union = len(idx1.union(idx2))
        jaccard_matrix[i, j] = intersection / union if union > 0 else 0.0

df_jaccard = pd.DataFrame(jaccard_matrix, index=[m.upper() for m in METHODS], columns=[m.upper() for m in METHODS])
print(f'Jaccard Overlap of Top-{best_n} Feature Sets:')
print(df_jaccard.to_string())

plt.figure(figsize=(6, 5))
sns.heatmap(df_jaccard, annot=True, cmap='Blues', fmt='.2f', vmin=0, vmax=1, cbar_kws={'label': 'Jaccard Index'})
plt.title(f'Jaccard Overlap of Top-{best_n} Feature Sets', fontsize=11)
plt.tight_layout()
jacc_path = os.path.join(FIGURES_DIR, '07_jaccard_overlap.png')
plt.savefig(jacc_path, dpi=120)
plt.close()
print(f'Jaccard heatmap saved -> {jacc_path}')


Jaccard Overlap of Top-25 Feature Sets:
           MRMR     ANOVA       RFE
MRMR   1.000000  0.428571  0.219512
ANOVA  0.428571  1.000000  0.351351
RFE    0.219512  0.351351  1.000000


Jaccard heatmap saved -> figures/07_jaccard_overlap.png


## Section 6 — Chosen Feature Set: Correlation Heatmap

In [8]:
best_indices, best_names = feature_selections[best_method][best_n]
df_corr = pd.DataFrame(X_tr[:, best_indices], columns=best_names)
corr_matrix = df_corr.corr().abs()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, cmap='Oranges', vmin=0, vmax=1,
            xticklabels=best_names, yticklabels=best_names, cbar_kws={'label': 'Absolute Correlation'})
plt.title(f'Feature Correlation Heatmap ({best_method.upper()}, n={best_n})', fontsize=12)

# Highlight high-correlation pairs (|r| > 0.8) with small markers or prints
print('High correlation pairs (|r| > 0.8) in chosen set:')
high_corr_found = False
for i in range(len(best_names)):
    for j in range(i+1, len(best_names)):
        r_val = corr_matrix.iloc[i, j]
        if r_val > 0.8:
            print(f'  {best_names[i]} <-> {best_names[j]} : {r_val:.3f}')
            high_corr_found = True
if not high_corr_found:
    print('  None found! Features are highly independent.')

plt.tight_layout()
corr_path = os.path.join(FIGURES_DIR, '07_feature_correlation.png')
plt.savefig(corr_path, dpi=120)
plt.close()
print(f'Correlation heatmap saved -> {corr_path}')


High correlation pairs (|r| > 0.8) in chosen set:
  od_R <-> color_hsv_c2_r3_mean : 0.812
  od_R <-> color_hed_c0_r3_mean : 0.955
  od_R <-> color_hed_c1_r3_mean : 0.958
  od_R <-> color_hed_c2_r3_mean : 0.959
  color_lab_c0_r7_std <-> color_lab_c0_r15_std : 0.816
  color_lab_c0_r7_std <-> color_hsv_c1_r7_std : 0.961
  color_lab_c0_r7_std <-> color_hsv_c2_r7_std : 0.993
  color_lab_c0_r7_std <-> color_hsv_c2_r15_std : 0.813
  color_lab_c0_r15_std <-> color_lab_c1_r15_std : 0.863
  color_lab_c0_r15_std <-> color_hsv_c1_r7_std : 0.817
  color_lab_c0_r15_std <-> color_hsv_c1_r15_std : 0.969
  color_lab_c0_r15_std <-> color_hsv_c2_r7_std : 0.809
  color_lab_c0_r15_std <-> color_hsv_c2_r15_std : 0.993
  color_lab_c1_r7_mean <-> color_lab_c1_r15_mean : 0.912
  color_lab_c1_r7_mean <-> color_lab_c2_r3_mean : 0.819
  color_lab_c1_r7_mean <-> color_lab_c2_r7_mean : 0.859
  color_lab_c1_r7_mean <-> color_hsv_c1_r3_mean : 0.852
  color_lab_c1_r7_mean <-> color_hsv_c1_r7_mean : 0.915
  color_lab_c

## Section 7 — Feature Group Contribution

In [9]:
FEATURE_GROUPS = {
    'OD (3)':             list(range(0, 3)),
    'Color stats (54)':   list(range(3, 57)),
    'LBP (3)':            list(range(57, 60)),
    'Gabor (12)':         list(range(60, 72)),
    'Gradient (5)':       list(range(72, 77)),
    'Struct tensor (3)':  list(range(77, 80)),
    'DoG (3)':            list(range(80, 83)),
    'Superpixel (2)':     list(range(83, 85)),
    'Entropy (1)':        [85],
    'Edge dist (1)':      [86],
    'GLCM (6)':           list(range(87, 93)),
}

group_counts = {}
for idx in best_indices:
    found_group = 'Unknown'
    for grp_name, indices_list in FEATURE_GROUPS.items():
        if idx in indices_list:
            found_group = grp_name
            break
    group_counts[found_group] = group_counts.get(found_group, 0) + 1

labels = list(group_counts.keys())
counts = list(group_counts.values())

plt.figure(figsize=(7, 7))
plt.pie(counts, labels=labels, autopct='%1.0f%%', startangle=140, colors=sns.color_palette('pastel', len(labels)))
plt.title(f'Feature Group Contribution (Top-{best_n} {best_method.upper()} features)', fontsize=11)

pie_path = os.path.join(FIGURES_DIR, '07_group_contribution.png')
plt.savefig(pie_path, dpi=120, bbox_inches='tight')
plt.close()
print(f'Pie chart saved -> {pie_path}')


Pie chart saved -> figures/07_group_contribution.png


## Section 8 — Decision & Save Selected Features

In [10]:
print('FEATURE SELECTION DECISION')
print(f'  Winner method: {best_method}')
print(f'  Chosen n: {best_n}')
print(f'  Validation macro-F1: {best_score:.4f}')
print(f'  Full 93 baseline F1: {baseline_f1:.4f}')

print('Saving selected features to JSON + CSV in data/models...')
out_paths = save_selected_features(
    indices=best_indices,
    names=best_names,
    method=best_method,
    n=best_n,
    out_dir=os.path.join(PROJECT_ROOT, 'data', 'models')
)

# Update DECISION.md
with open(decision_md_path, 'r') as fh:
    content = fh.read()

table_rows = ''
for n in N_VALUES:
    mrmr_f1  = df_sweep[(df_sweep['method']=='mrmr')  & (df_sweep['n']==n)]['macro_f1'].values[0]
    anova_f1 = df_sweep[(df_sweep['method']=='anova') & (df_sweep['n']==n)]['macro_f1'].values[0]
    rfe_f1   = df_sweep[(df_sweep['method']=='rfe')   & (df_sweep['n']==n)]['macro_f1'].values[0]
    
    if n == 10:
        delta_str = '—'
    else:
        # Delta of winning method from n-5
        prev_f1 = df_sweep[(df_sweep['method'] == best_method) & (df_sweep['n'] == n-5)]['macro_f1'].values[0]
        curr_f1 = df_sweep[(df_sweep['method'] == best_method) & (df_sweep['n'] == n)]['macro_f1'].values[0]
        delta_str = f'{curr_f1 - prev_f1:+.4f}'
        
    mrmr_str  = f'**{mrmr_f1:.4f}**'  if best_method == 'mrmr'  and n == best_n else f'{mrmr_f1:.4f}'
    anova_str = f'**{anova_f1:.4f}**' if best_method == 'anova' and n == best_n else f'{anova_f1:.4f}'
    rfe_str   = f'**{rfe_f1:.4f}**'   if best_method == 'rfe'   and n == best_n else f'{rfe_f1:.4f}'
    
    table_rows += f'| {n} | {mrmr_str} | {anova_str} | {rfe_str} | {delta_str} |\n'

new_table = (
    '| n  | mRMR F1 | ANOVA F1 | RFE F1 | Delta from n-5 |\n'
    '|----|---------|---------|--------|----------------|\n'
    + table_rows.rstrip('\n')
)

# Replace feature sweep table block
old_table_pattern = r'(\*\*Rule:\*\*.*?\n\n)\| n  \| mRMR.*?(?=\n\n)'
replacement_block = r'\g<1>' + new_table
content = re.sub(old_table_pattern, replacement_block, content, flags=re.DOTALL)

# Replace status line
content = content.replace(
    '**Status:** PENDING — to be filled after `notebooks/07_inspect_feature_selection.ipynb` runs.',
    '**Status:** CONFIRMED — filled by `notebooks/07_inspect_feature_selection.ipynb`.'
)

# Replace final choice placeholders
content = content.replace(
    '**Final n:** `[TO BE FILLED BY NOTEBOOK]`',
    f'**Final n:** `{best_n}`'
)
content = content.replace(
    '**Final method:** `[TO BE FILLED BY NOTEBOOK]`',
    f'**Final method:** `{best_method}`'
)

with open(decision_md_path, 'w') as fh:
    fh.write(content)

print(f'DECISION.md updated successfully at: {decision_md_path}')
# Print Decision 2 section
lines = content.split('\n')
in_d2 = False
for line in lines:
    if '## Decision 2' in line:
        in_d2 = True
    if in_d2:
        print(line)

# Write a results json
fs_results = {
    'winner_method': best_method,
    'chosen_n': best_n,
    'validation_macro_f1': best_score,
    'baseline_93_f1': baseline_f1,
    'sweep_results': sweep_results,
    'out_paths': out_paths,
}
fs_results_path = os.path.join(PHASE_DIR, 'feature_selection_results.json')
with open(fs_results_path, 'w') as fh:
    json.dump(fs_results, fh, indent=2)
print(f'feature_selection_results.json written -> {fs_results_path}')


FEATURE SELECTION DECISION
  Winner method: rfe
  Chosen n: 25
  Validation macro-F1: 0.8681
  Full 93 baseline F1: 0.8699
Saving selected features to JSON + CSV in data/models...
  ✓ Feature list written: /run/media/mananbyte/newvol/Pannuke-project/data/models/selected_features_rfe_25.json
  ✓ Feature CSV written:  /run/media/mananbyte/newvol/Pannuke-project/data/models/selected_features_rfe_25.csv
DECISION.md updated successfully at: /run/media/mananbyte/newvol/Pannuke-project/.planning/phases/03-dataset-standardization-mrmr-selection/DECISION.md
## Decision 2: Feature Count `n`

**Status:** CONFIRMED — filled by `notebooks/07_inspect_feature_selection.ipynb`.

**Method:** F1-elbow analysis over `n ∈ {10, 15, 20, 25, 30}`.

**Rule:** Choose the smallest `n` where the marginal F1 gain from adding the next 5 features  
drops below **0.002** (elbow point). Hard cap at 30.

| n  | mRMR F1 | ANOVA F1 | RFE F1 | Delta from n-5 |
|----|---------|---------|--------|----------------|
| 10 | 0